## Creating the Notebook and importing in Data from the Group Github

In [1]:
#Import Initial libraries

#Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile



In [2]:
#Run code provided by Chance for consitent notebooks within the group

#File path to the zipped datafile
zip_path = '../../Data/Processed/Data_Compressed.zip'

#List of filenames to read from the zip file
filenames = [
    'Data_Compressed/all_normalized_features.csv',
    'Data_Compressed/kdd_all_normalized_features.csv',
    'Data_Compressed/kdd_expanded_all_scaled.csv',
    'Data_Compressed/kdd_merged_normalized_all.csv'
]

#Corresponding names for the DataFrames
df_names = [
    'xuetangx_df',
    'kdd_df',
    'kdd_expanded_df',
    'kdd_merged_df'
]

#Create an empty dictionary to store the DataFrames
dataframes = {}

#Open the zip file
z = zipfile.ZipFile(zip_path, 'r')

#Loop through the filenames and read them into DataFrames
for i, file in enumerate(filenames):
    print(f"Reading file: {file} from {zip_path}")
    
    #Read each CSV file directly from the zip
    f = z.open(file)
    dataframes[df_names[i]] = pd.read_csv(f)
    f.close()
    
    print(f"{df_names[i]} loaded with shape: {dataframes[df_names[i]].shape}\n")

#Close the zip file after reading
z.close()

#Assign individual DataFrames to variables
xuetangx_df = dataframes['xuetangx_df']
kdd_df = dataframes['kdd_df']
kdd_expanded_df = dataframes['kdd_expanded_df']
kdd_merged_df = dataframes['kdd_merged_df']

Reading file: Data_Compressed/all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
xuetangx_df loaded with shape: (225642, 30)

Reading file: Data_Compressed/kdd_all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
kdd_df loaded with shape: (200904, 17)

Reading file: Data_Compressed/kdd_expanded_all_scaled.csv from ../../Data/Processed/Data_Compressed.zip
kdd_expanded_df loaded with shape: (120542, 142)

Reading file: Data_Compressed/kdd_merged_normalized_all.csv from ../../Data/Processed/Data_Compressed.zip
kdd_merged_df loaded with shape: (120542, 158)



## Reproduction of the 2nd Experiment with Machine Learning Models
- Use larger feature set (140 features) generarated by Peng and Aggarwal
  - kdd_expanded_df

In [3]:
#View the basic info for the data set before starting toensure proper importation

kdd_expanded_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120542 entries, 0 to 120541
Data columns (total 142 columns):
 #    Column                             Dtype  
---   ------                             -----  
 0    Unnamed: 0                         int64  
 1    enrollment_id                      int64  
 2    truth                              int64  
 3    avg_chapter_delays                 float64
 4    server_discussion_percent          int64  
 5    act_cnt_weekDay_01                 float64
 6    browser_html_percent               int64  
 7    parallel_enrollments               float64
 8    browser_dictation                  int64  
 9    act_cnt_day_00                     int64  
 10   act_cnt_day_01                     float64
 11   act_cnt_day_02                     float64
 12   act_cnt_day_03                     float64
 13   act_cnt_day_04                     float64
 14   act_cnt_day_05                     float64
 15   act_cnt_day_06                     float64
 16   

In [4]:
#View header information for Kdd-expanded
kdd_expanded_df.head()

,Unnamed: 0,enrollment_id,truth,avg_chapter_delays,server_discussion_percent,act_cnt_weekDay_01,browser_html_percent,parallel_enrollments,browser_dictation,act_cnt_day_00,...,server_course_percent,browser_course_info_percent,browser_course,browser_vertical_percent,sessions_in_week_1,sessions_in_week_0,sessions_in_week_3,sessions_in_week_2,sessions_in_week_4,browser_about
0,0,1,0,0.401837,0,0.317621,0,3.960922,0,0,...,0,0,0,0,2.076406,-0.029110,1.685275,2.636167,2.141673,0
1,1,3,0,0.460054,0,-0.005913,0,-0.449933,0,0,...,0,0,0,0,0.565552,0.466433,0.126527,0.087058,0.928388,0
2,2,4,0,0.866122,0,-0.094149,0,0.652780,0,0,...,0,0,0,0,1.572788,1.457518,-0.393056,2.126345,-0.284897,0
3,3,5,0,0.429975,0,1.964698,0,-0.449933,0,0,...,0,0,0,0,7.616205,0.466433,6.361519,0.087058,4.568243,0
4,4,6,0,-0.255399,0,-0.123561,0,-0.449933,0,0,...,0,0,0,0,-0.441684,-0.524652,0.126527,-0.422764,2.141673,0


### Importing Additional Libraries for modeling

In [13]:
# ImportModels based on the Jupyter Notebook on Github related to the paper and modeling experiments
## https://github.com/rambasnet/PredictingMOOCDropouts/blob/master/KDD_experiments.ipynb

# Additional Scikit-Learn imports
from sklearn.model_selection import StratifiedKFold

# Scikit-Learn's ML models
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

# Had assistance from the Anaconda AI Assistant due to needing to pip install the library first
import xgboost as xgb
from xgboost import XGBClassifier

#Importing additional pieces of sklearn that may be needed 
## These imports are based on modeling in DATA6320
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, roc_auc_score

In [14]:
#Create the X and Y data sets
# 'truth' is the variable for dropout rate as seen in the original code from the research
y= kdd_expanded_df['truth']
display(y.value_counts())
display(round(y.value_counts(normalize=True),3))

truth
1    95581
0    24961
Name: count, dtype: int64

truth
1    0.793
0    0.207
Name: proportion, dtype: float64

In [15]:
#Create the X data set with by dropping columns that do not belong in the model BurgStatus

X = kdd_expanded_df.drop (['truth','Unnamed: 0','enrollment_id'], axis=1)
X.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120542 entries, 0 to 120541
Data columns (total 139 columns):
 #    Column                             Dtype  
---   ------                             -----  
 0    avg_chapter_delays                 float64
 1    server_discussion_percent          int64  
 2    act_cnt_weekDay_01                 float64
 3    browser_html_percent               int64  
 4    parallel_enrollments               float64
 5    browser_dictation                  int64  
 6    act_cnt_day_00                     int64  
 7    act_cnt_day_01                     float64
 8    act_cnt_day_02                     float64
 9    act_cnt_day_03                     float64
 10   act_cnt_day_04                     float64
 11   act_cnt_day_05                     float64
 12   act_cnt_day_06                     float64
 13   act_cnt_day_07                     float64
 14   act_cnt_day_08                     float64
 15   act_cnt_day_09                     float64
 16   

In [16]:
### Split the data set into an 80/20 random split for training and testing
#Creating the training data using train test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 100)


In [17]:
# Create function for short train test to condense the output 
## Technique from Data 6320

def shorttraintest(vartrain, vartest, y_train, y_test, model):

    #Fit the model
    model.fit(vartrain, y_train)

    #Predict with the model
    model_pred = model.predict(vartest)
    model_prob = model.predict_proba(vartest)


    print('Confusion Matrix:')
    print(confusion_matrix(y_test, model_pred))
    print("")

    #Assess with the model
    score = model.score(vartest, y_test)
    score_format = 'Accuracy Score: {0:.4f}'.format(score)
    print(score_format)

    recall = recall_score(y_test, model_pred)
    recall_format = 'Recall Score: {0:.4f}'.format(recall)
    print(recall_format)
    
    precision = precision_score(y_test, model_pred)
    precision_format = 'Precision Score: {0:.4f}'.format(precision)
    print(precision_format)
    
    # calculate roc curve
    y_pred_prob = model.predict_proba(vartest)[:,1]
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    roc_auc_format = 'ROC AUC Score: {0:.4f}'.format(roc_auc)
    print(roc_auc_format)
    print('')

### Model 1: Random Forest (RF)

In [18]:
#Running the Random Forest Model

vartrain = X_train
vartest = X_test
model = RandomForestClassifier()
shorttraintest(vartrain, vartest, y_train, y_test, model)

Confusion Matrix:
[[ 2911  2090]
 [  873 18235]]

Accuracy Score: 0.8771
Recall Score: 0.9543
Precision Score: 0.8972
ROC AUC Score: 0.8763



In [19]:
#Conducting 5 fold Cross Validation for Random Forest

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
cv_model = RandomForestClassifier() 

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')

KeyboardInterrupt: 

### Model 2: XGBoost (XGB)

In [ ]:
#Running the XGB Model
vartrain = X_train
vartest = X_test
model =  XGBClassifier()

shorttraintest(vartrain, vartest, y_train, y_test, model)

Confusion Matrix:
[[ 2854  2147]
 [  856 18252]]

Accuracy Score: 0.8754
Recall Score: 0.9552
Precision Score: 0.8947
ROC AUC Score: 0.8812



In [ ]:
#Conducting 5 fold Cross Validation for XGB

cv_model = XGBClassifier()

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')


Cross-validation scores: [0.87281589 0.87006792 0.87530461 0.87695738 0.87410557]
Mean CV accuracy: 0.8739
Standard deviation: 0.0023
CV Accuracy Scores:
[0.87281589 0.87006792 0.87530461 0.87695738 0.87410557]



### Model 3: AdaBoost Classifier (AB)

In [ ]:
#Running the AdaBoost Classifier
vartrain = X_train
vartest = X_test
model = AdaBoostClassifier()

shorttraintest(vartrain, vartest, y_train, y_test, model)

c:\Users\autum\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Confusion Matrix:
[[ 2818  2183]
 [  754 18354]]

Accuracy Score: 0.8782
Recall Score: 0.9605
Precision Score: 0.8937
ROC AUC Score: 0.8803



In [ ]:
#Running 5 fold Cross Validation for AB
cv_model = AdaBoostClassifier()

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')

c:\Users\autum\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\autum\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\autum\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\autum\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\aut

Cross-validation scores: [0.87514906 0.87395655 0.87810442 0.87623146 0.87685368]
Mean CV accuracy: 0.8761
Standard deviation: 0.0014
CV Accuracy Scores:
[0.87514906 0.87395655 0.87810442 0.87623146 0.87685368]



### Model 4: Decision Tree (DT)

In [ ]:
#Running the DecisionTree Classifier
vartrain = X_train
vartest = X_test
model = DecisionTreeClassifier()

shorttraintest(vartrain, vartest, y_train, y_test, model)

Confusion Matrix:
[[ 2777  2224]
 [ 2405 16703]]

Accuracy Score: 0.8080
Recall Score: 0.8741
Precision Score: 0.8825
ROC AUC Score: 0.7149



In [ ]:
#Running 5 fold Cross Validation for DT

cv_model = DecisionTreeClassifier()

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')

Cross-validation scores: [0.80188728 0.80655364 0.80764245 0.80939542 0.8037955 ]
Mean CV accuracy: 0.8059
Standard deviation: 0.0027
CV Accuracy Scores:
[0.80188728 0.80655364 0.80764245 0.80939542 0.8037955 ]



### Model 5: K-Neighbors (kNN)

In [ ]:
#Running the K-Neighbors Model
vartrain = X_train
vartest = X_test
model = KNeighborsClassifier()
shorttraintest(vartrain, vartest, y_train, y_test, model)

Confusion Matrix:
[[ 2249  2752]
 [  710 18398]]

Accuracy Score: 0.8564
Recall Score: 0.9628
Precision Score: 0.8699
ROC AUC Score: 0.8019



In [ ]:
#Running 5 fold Cross Validation for kNN

cv_model =  KNeighborsClassifier()

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')

Cross-validation scores: [0.85243947 0.85218023 0.85243947 0.85372809 0.85367624]
Mean CV accuracy: 0.8529
Standard deviation: 0.0007
CV Accuracy Scores:
[0.85243947 0.85218023 0.85243947 0.85372809 0.85367624]



### Model 6: Logistic Regression (LR)

In [ ]:
#Running 5 fold Cross Validation LR
vartrain = X_train
vartest = X_test
model = LogisticRegression()

shorttraintest(vartrain, vartest, y_train, y_test, model)

Confusion Matrix:
[[ 2585  2416]
 [  655 18453]]

Accuracy Score: 0.8726
Recall Score: 0.9657
Precision Score: 0.8842
ROC AUC Score: 0.8783



In [ ]:
#Running 5 fold Cross Validation for LR

cv_model =  LogisticRegression()

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')

Cross-validation scores: [0.87183077 0.86980868 0.87597864 0.8732241  0.87296484]
Mean CV accuracy: 0.8728
Standard deviation: 0.0020
CV Accuracy Scores:
[0.87183077 0.86980868 0.87597864 0.8732241  0.87296484]



### Model 7: Linear Discriminate Analysis (LDA)

In [ ]:
#Running the Model for LDA

vartrain = X_train
vartest = X_test
model =LinearDiscriminantAnalysis()

shorttraintest(vartrain, vartest, y_train, y_test, model)

Confusion Matrix:
[[ 2462  2539]
 [  593 18515]]

Accuracy Score: 0.8701
Recall Score: 0.9690
Precision Score: 0.8794
ROC AUC Score: 0.8726



In [ ]:
#Running 5 fold Cross Validation for LDA

cv_model = LinearDiscriminantAnalysis()

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')

Cross-validation scores: [0.86830508 0.86664593 0.87364546 0.86959452 0.87016489]
Mean CV accuracy: 0.8697
Standard deviation: 0.0023
CV Accuracy Scores:
[0.86830508 0.86664593 0.87364546 0.86959452 0.87016489]



### Model 8: Naïve Bayes Classifier (NB)

In [ ]:
#Running the Model for NB

vartrain = X_train
vartest = X_test
model = GaussianNB()

shorttraintest(vartrain, vartest, y_train, y_test, model)

Confusion Matrix:
[[ 2763  2238]
 [ 1381 17727]]

Accuracy Score: 0.8499
Recall Score: 0.9277
Precision Score: 0.8879
ROC AUC Score: 0.7737



In [ ]:
#Running 5 fold Cross Validation for NB

cv_model =  GaussianNB()

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train, cv=5, scoring='accuracy')

#Printing cv scores, mean and standard deviation for the model
##Co-pilot in VS Code assisted with code for printing mean and standard deviation
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")
print('CV Accuracy Scores:')
print(cv_scores)
print('')

Cross-validation scores: [0.84839529 0.8480842  0.85487634 0.8514985  0.85253552]
Mean CV accuracy: 0.8511
Standard deviation: 0.0026
CV Accuracy Scores:
[0.84839529 0.8480842  0.85487634 0.8514985  0.85253552]



### Model 9: Support Vector Machine (SVM)
- Running this model last due to computational power needed
- Sampling a subset of the data due to run time of the full data creating issues and timing out

In [ ]:
#Running model with a sample of the data for managability
##Used Anaconda Assistant for help with figuring out my run time issue and providing the sampling code
vartrain = X_train.sample(frac=0.1, random_state=42)
vartest = X_test
model = SVC()

shorttraintest(vartrain, vartest, y_train, y_test, model)

ValueError: Found input variables with inconsistent numbers of samples: [9643, 96433]

In [ ]:
#Entered Output Error into Anaconda Assistant and am running the code provided

# Define input variables - using only 10% of training data to make computation manageable
# Sample both X_train and y_train with the same indices to maintain alignment
train_indices = X_train.sample(frac=0.1, random_state=42).index
vartrain = X_train.loc[train_indices]
y_train_sampled = y_train.loc[train_indices]  # Assuming y_train is a pandas Series/DataFrame

# Keep test data as is
vartest = X_test

# Initialize Support Vector Machine Classifier
model = SVC()

# Use the sampled training data with corresponding sampled labels
shorttraintest(vartrain, vartest, y_train_sampled, y_test, model)

AttributeError: This 'SVC' has no attribute 'predict_proba'

In [ ]:
#Entered Output Error into Anaconda Assistant and am running the code provided

# Define input variables - using only 10% of training data to make computation manageable
# Sample both X_train and y_train with the same indices to maintain alignment
train_indices = X_train.sample(frac=0.1, random_state=42).index
vartrain = X_train.loc[train_indices]
y_train_sampled = y_train.loc[train_indices]  # Assuming y_train is a pandas Series/DataFrame

# Keep test data as is
vartest = X_test

# Initialize Support Vector Machine Classifier with probability=True
# This enables the predict_proba method that the shorttraintest function is trying to use
model = SVC(probability=True)

# Use the sampled training data with corresponding sampled labels
shorttraintest(vartrain, vartest, y_train_sampled, y_test, model)

Confusion Matrix:
[[ 2604  2397]
 [  813 18295]]

Accuracy Score: 0.8669
Recall Score: 0.9575
Precision Score: 0.8842
ROC AUC Score: 0.8298



In [ ]:
# Cross-validation on the sampled data
#Created with the Anaconda Assistant to ensure(hopefully) proper execution of  cross validation with sample data

# Define the model for cross-validation
cv_model = SVC(probability=True)

# Perform k-fold cross-validation (using 5 folds)
cv_scores = cross_val_score(cv_model, vartrain, y_train_sampled, cv=5, scoring='accuracy')

# Print cross-validation results
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")


Cross-validation scores: [0.85743909 0.86832556 0.8750648  0.86255187 0.8620332 ]
Mean CV accuracy: 0.8651
Standard deviation: 0.0061


In [ ]:
#Creating a Dataframe for all modelw
##Created with co-pilot in VS Code

# Define a dictionary to store the results
cv_results = {
    "Model": [],
    "CV Scores": [],
    "Mean Accuracy": [],
    "Standard Deviation": []
}

# Add results for each model (example for Random Forest)
cv_results["Model"].append("Random Forest")
cv_results["CV Scores"].append(cv_scores)  # Use the cv_scores from your Random Forest cross-validation
cv_results["Mean Accuracy"].append(cv_scores.mean())
cv_results["Standard Deviation"].append(cv_scores.std())

# Repeat for other models...

# Create a DataFrame from the results
cv_results_df = pd.DataFrame(cv_results)

# Display the DataFrame
cv_results_df

,Model,CV Scores,Mean Accuracy,Standard Deviation
0,Random Forest,"[0.8574390876101607, 0.8683255572835666, 0.875...",0.865083,0.006071
